[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/05_one_vs_all.ipynb)

# Bài 05 — Phân loại nhiều lớp với Logistic Regression: One-vs-All

## 1. One-vs-All (One-vs-Rest)

Ý tưởng: đưa bài toán $K$ lớp về $K$ bài toán **nhị phân**. Với lớp $k$: gán nhãn 1 cho các mẫu thuộc lớp $k$, nhãn 0 cho tất cả các lớp còn lại, rồi train một Logistic Regression $h^{(k)}_\theta(x) \approx P(y = k \mid x)$. Khi dự đoán, chọn lớp có xác suất cao nhất:

$$\hat y = \arg\max_k h^{(k)}_\theta(x)$$

<img src="images/one-vs-all.png" style="width:45%;height:45%;">

<img src="images/one-vs-all-3.jpg" style="width:45%;height:45%;">

> 📝 *So với bản gốc:* bản gốc minh hoạ trên CIFAR-10 (ảnh màu 32×32, ~170 MB, phải tải về và dùng code phụ trợ riêng). Ở đây dùng bộ `digits` 10 lớp có sẵn offline và **tự cài đặt** One-vs-All từ class `LogisticRegressionScratch` của khoá học. Phần CIFAR-10 của bản gốc được làm đầy đủ — cùng softmax và SVM đa lớp — ở [bài 08](08_linear_classifiers_cifar10.ipynb).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

from ml_course.algorithms import LogisticRegressionScratch
from ml_course.datasets import load_digits_data
from ml_course.visualization import plot_confusion_matrix, set_style

set_style()
X, y, images = load_digits_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_test.shape, "| số lớp:", len(np.unique(y)))

## 2. Cài đặt One-vs-All from scratch

In [ ]:
class OneVsAll:
    def __init__(self, **logreg_params):
        self.logreg_params = logreg_params

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.models_ = []
        for k in self.classes_:
            y_binary = (y == k).astype(float)        # lớp k → 1, các lớp khác → 0
            self.models_.append(LogisticRegressionScratch(**self.logreg_params).fit(X, y_binary))
        return self

    def decision_scores(self, X):
        # cột k = P(y = k | x) theo classifier thứ k
        return np.column_stack([m.predict_proba(X)[:, 1] for m in self.models_])

    def predict(self, X):
        return self.classes_[np.argmax(self.decision_scores(X), axis=1)]


ova = OneVsAll(lr=0.5, n_iters=1500, l2=1.0).fit(X_train, y_train)
pred = ova.predict(X_test)
print(f"Accuracy (One-vs-All from scratch): {np.mean(pred == y_test):.4f}")

**Chú ý:** các "xác suất" của $K$ classifier được train **độc lập** nên tổng của chúng **không** bằng 1:

In [ ]:
scores = ova.decision_scores(X_test[:3])
print(scores.round(3))
print("Tổng mỗi hàng:", scores.sum(axis=1).round(3))

## 3. Confusion matrix và các lỗi thường gặp

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
plot_confusion_matrix(cm, labels=list(range(10)), ax=ax, title="One-vs-All trên digits")
plt.show()

## 4. Trọng số của từng classifier = "ảnh mẫu" của từng chữ số *(bổ sung)*

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for k, ax in enumerate(axes):
    w = ova.models_[k].coef_.reshape(8, 8)
    lim = np.abs(w).max()
    ax.imshow(w, cmap="RdBu", vmin=-lim, vmax=lim)
    ax.set_title(f"lớp {k}")
    ax.axis("off")
fig.suptitle("Trọng số của 10 classifier (xanh = ủng hộ lớp đó)", y=1.08)
plt.show()

## 5. scikit-learn: OvR và multinomial

`sklearn.linear_model.LogisticRegression` với nhiều lớp mặc định dùng **multinomial** (Softmax Regression — bài 06). Có thể ép dùng One-vs-Rest bằng `OneVsRestClassifier`:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

ovr_sk = OneVsRestClassifier(LogisticRegression(max_iter=3000)).fit(X_train, y_train)
multi_sk = LogisticRegression(max_iter=3000).fit(X_train, y_train)
print(f"sklearn One-vs-Rest : {ovr_sk.score(X_test, y_test):.4f}")
print(f"sklearn multinomial : {multi_sk.score(X_test, y_test):.4f}")

## 6. Nhận xét

- One-vs-All đơn giản, dễ cài đặt, song song hoá được ($K$ model độc lập), và **vẫn được dùng nhiều** (vd: `liblinear`, nhiều bài toán phân loại văn bản).
- Nhược điểm: các classifier không "biết" về nhau → điểm số không phải phân phối xác suất (tổng ≠ 1), có thể không nhất quán; mỗi classifier gặp dữ liệu **mất cân bằng** (1 lớp vs $K - 1$ lớp).
- **Softmax Regression** (bài 06) học $K$ lớp **cùng lúc**, cho xác suất hợp lệ — về độ chính xác, hai cách thường **gần nhau** (như kết quả trên), nhưng softmax là nền tảng của lớp output trong mọi neural network phân loại.

> 📝 *Đính chính:* bản gốc nhận định One-vs-All "cực ít được dùng" và kém hơn softmax "rất nhiều" — thực nghiệm trên cho thấy chênh lệch thường nhỏ; kết quả ~32% của bản gốc trên CIFAR-10 chủ yếu do Logistic Regression trên pixel thô quá yếu cho ảnh màu phức tạp, không phải do One-vs-All.

## 🧠 Kiểm tra nhanh

1. Với 10 lớp, One-vs-All train bao nhiêu classifier nhị phân? Mỗi classifier thấy tỉ lệ dương/âm thế nào?
2. Vì sao tổng "xác suất" của các classifier One-vs-All không bằng 1?
3. Khi nào One-vs-All có lợi thế so với softmax?

## Tài liệu tham khảo

[1] [CS231n: Linear classification](https://cs231n.github.io/linear-classify/) · [2] scikit-learn — [Multiclass and multioutput algorithms](https://scikit-learn.org/stable/modules/multiclass.html)

---
<sub>Bài gốc: *Logistic Regression — One vs All* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License). Đã thay CIFAR-10 bằng digits offline, cài đặt One-vs-All from scratch, bổ sung trực quan hoá trọng số và so sánh với multinomial.</sub>